In [3]:
import pandas as pd
import numpy as np
from math import radians, sin, cos, sqrt, atan2

df = pd.read_csv('../data/raw/hdb_resale_transactions.csv')

geocoded = pd.read_csv('../data/processed/geocoded_addresses.csv')

print(f"Transactions: {len(df):,}")
print(f"Geocoded addresses: {len(geocoded):,}")
df.head()

Transactions: 233,055
Geocoded addresses: 9,712


,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,remaining_lease,resale_price
0,2017-01,ANG MO KIO,2 ROOM,406,ANG MO KIO AVE 10,10 TO 12,44.0,Improved,1979,61 years 04 months,232000.0
1,2017-01,ANG MO KIO,3 ROOM,108,ANG MO KIO AVE 4,01 TO 03,67.0,New Generation,1978,60 years 07 months,250000.0
2,2017-01,ANG MO KIO,3 ROOM,602,ANG MO KIO AVE 5,01 TO 03,67.0,New Generation,1980,62 years 05 months,262000.0
3,2017-01,ANG MO KIO,3 ROOM,465,ANG MO KIO AVE 10,04 TO 06,68.0,New Generation,1980,62 years 01 month,265000.0
4,2017-01,ANG MO KIO,3 ROOM,601,ANG MO KIO AVE 5,01 TO 03,67.0,New Generation,1980,62 years 05 months,265000.0


In [4]:
df = df.merge(geocoded, on=['block', 'street_name'], how='left')

print(f"Transactions after merge: {len(df):,}")
print(f"Missing coordinates: {df['latitude'].isna().sum():,}")
print(f"Coverage: {(df['latitude'].notna().sum() / len(df) * 100):.1f}%")

Transactions after merge: 233,055
Missing coordinates: 0
Coverage: 100.0%


In [17]:
mrt = pd.read_csv('../data/processed/mrt_stations_geocoded.csv')
mrt['opening_date'] = pd.to_datetime(mrt['opening_date'])

schools = pd.read_csv('../data/processed/schools_geocoded.csv')
hawkers = pd.read_csv('../data/processed/hawker_centres.csv')
hawkers['est_completion_date'] = pd.to_datetime(hawkers['est_completion_date'], errors='coerce')
malls = pd.read_csv('../data/processed/malls_geocoded.csv')
expressways = pd.read_csv('../data/processed/expressway_coords.csv')
bus_stops = pd.read_csv('../data/processed/bus_stops.csv')

print(f"MRT stations:   {len(mrt):,}")
print(f"Schools:        {len(schools):,}")
print(f"Hawker centres: {len(hawkers):,}")
print(f"Malls:          {len(malls):,}")
print(f"Expressway pts: {len(expressways):,}")
print(f"Bus stops:      {len(bus_stops):,}")

MRT stations:   143
Schools:        337
Hawker centres: 123
Malls:          116
Expressway pts: 3,319
Bus stops:      5,166


In [8]:
def haversine(lat1, lon1, lat2, lon2):
    """
    Calculate the great-circle distance between two points on Earth in kilometres.
    Uses the Haversine formula which accounts for Earth's curvature.
    """
    R = 6371  # Earth's radius in kilometres

    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = sin(dlat/2)**2 + cos(lat1) * cos(lat2) * sin(dlon/2)**2
    c = 2 * atan2(sqrt(a), sqrt(1-a))

    return R * c

In [12]:
def dist_to_amenities(lat, lon, amenity_df, lat_col='latitude', lon_col='longitude'):
    """
    Calculate distances from a single point to all rows in amenity_df.
    Returns a numpy array of distances in kilometres.
    """
    lat_rad = radians(lat)
    lon_rad = radians(lon)
    
    amenity_lats = np.radians(amenity_df[lat_col].values)
    amenity_lons = np.radians(amenity_df[lon_col].values)
    
    dlat = amenity_lats - lat_rad
    dlon = amenity_lons - lon_rad
    
    a = np.sin(dlat/2)**2 + np.cos(lat_rad) * np.cos(amenity_lats) * np.sin(dlon/2)**2
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1-a))
    
    return 6371 * c

# Test on first transaction
lat, lon = df.iloc[0]['latitude'], df.iloc[0]['longitude']
distances = dist_to_amenities(lat, lon, mrt)
print(f"Distance to nearest MRT: {distances.min():.3f} km")
print(f"MRT stations within 1km: {(distances < 1).sum()}")

Distance to nearest MRT: 1.004 km
MRT stations within 1km: 0


In [13]:
# Convert transaction month to datetime for date comparisons
df['transaction_date'] = pd.to_datetime(df['month'])

# Test opening date filter on first transaction
transaction_date = df.iloc[0]['transaction_date']
mrt_available = mrt[mrt['opening_date'] <= transaction_date]
print(f"Transaction date: {transaction_date.date()}")
print(f"MRT stations available at that date: {len(mrt_available)}")
print(f"Total MRT stations: {len(mrt)}")
print(f"Stations not yet open: {len(mrt) - len(mrt_available)}")

Transaction date: 2017-01-01
MRT stations available at that date: 102
Total MRT stations: 143
Stations not yet open: 41


In [22]:
# Initialise result lists
dist_nearest_mrt = []
num_mrt_within_1km = []
num_mrt_within_2km = []
dist_nearest_school = []
num_schools_within_1km = []
dist_nearest_primary_school = []
num_primary_schools_within_1km = []
dist_nearest_mall = []
num_malls_within_2km = []
dist_nearest_hawker = []
num_hawkers_within_500m = []
dist_to_cbd = []
dist_nearest_expressway = []
dist_nearest_bus_stop = []
num_bus_stops_within_300m = []

# CBD coordinates — Raffles Place
CBD_LAT, CBD_LON = 1.2830, 103.8513

# Primary school filter
primary_schools = schools[schools['mainlevel_code'].isin(['PRIMARY', 'MIXED LEVEL (P1-S4)'])].copy()

print(f"Primary schools: {len(primary_schools)}")
print(f"All schools: {len(schools)}")

Primary schools: 182
All schools: 337


In [23]:
for idx, row in df.iterrows():
    lat, lon = row['latitude'], row['longitude']
    transaction_date = row['transaction_date']

    # MRT — filter by opening date
    mrt_available = mrt[mrt['opening_date'] <= transaction_date]
    mrt_dists = dist_to_amenities(lat, lon, mrt_available)
    dist_nearest_mrt.append(mrt_dists.min() if len(mrt_dists) > 0 else np.nan)
    num_mrt_within_1km.append((mrt_dists < 1).sum())
    num_mrt_within_2km.append((mrt_dists < 2).sum())

    # Schools
    school_dists = dist_to_amenities(lat, lon, schools)
    dist_nearest_school.append(school_dists.min())
    num_schools_within_1km.append((school_dists < 1).sum())

    # Primary schools
    pri_dists = dist_to_amenities(lat, lon, primary_schools)
    dist_nearest_primary_school.append(pri_dists.min())
    num_primary_schools_within_1km.append((pri_dists < 1).sum())

    # Malls
    mall_dists = dist_to_amenities(lat, lon, malls)
    dist_nearest_mall.append(mall_dists.min())
    num_malls_within_2km.append((mall_dists < 2).sum())

    # Hawker centres — filter by completion date
    hawkers_available = hawkers[
        hawkers['est_completion_date'].isna() | 
        (hawkers['est_completion_date'] <= transaction_date)
    ]
    hawker_dists = dist_to_amenities(lat, lon, hawkers_available)
    dist_nearest_hawker.append(hawker_dists.min() if len(hawker_dists) > 0 else np.nan)
    num_hawkers_within_500m.append((hawker_dists < 0.5).sum())

    # CBD distance
    dist_to_cbd.append(haversine(lat, lon, CBD_LAT, CBD_LON))

    # Expressways
    exp_dists = dist_to_amenities(lat, lon, expressways)
    dist_nearest_expressway.append(exp_dists.min())

    # Bus stops
    bus_dists = dist_to_amenities(lat, lon, bus_stops)
    dist_nearest_bus_stop.append(bus_dists.min())
    num_bus_stops_within_300m.append((bus_dists < 0.3).sum())

    if (idx + 1) % 10000 == 0:
        print(f"Processed {idx + 1:,} / {len(df):,}")

print("Done.")

Processed 10,000 / 233,055
Processed 20,000 / 233,055
Processed 30,000 / 233,055
Processed 40,000 / 233,055
Processed 50,000 / 233,055
Processed 60,000 / 233,055
Processed 70,000 / 233,055
Processed 80,000 / 233,055
Processed 90,000 / 233,055
Processed 100,000 / 233,055
Processed 110,000 / 233,055
Processed 120,000 / 233,055
Processed 130,000 / 233,055
Processed 140,000 / 233,055
Processed 150,000 / 233,055
Processed 160,000 / 233,055
Processed 170,000 / 233,055
Processed 180,000 / 233,055
Processed 190,000 / 233,055
Processed 200,000 / 233,055
Processed 210,000 / 233,055
Processed 220,000 / 233,055
Processed 230,000 / 233,055
Done.


In [24]:
print(f"Length of results: {len(dist_nearest_mrt):,}")
print(f"\nSample values:")
print(f"dist_nearest_mrt:       {dist_nearest_mrt[:3]}")
print(f"dist_nearest_hawker:    {dist_nearest_hawker[:3]}")
print(f"dist_to_cbd:            {dist_to_cbd[:3]}")
print(f"num_mrt_within_1km:     {num_mrt_within_1km[:3]}")
print(f"num_hawkers_within_500m:{num_hawkers_within_500m[:3]}")

Length of results: 233,055

Sample values:
dist_nearest_mrt:       [np.float64(1.0039959848633804), np.float64(1.267605333327398), np.float64(1.0711775793770115)]
dist_nearest_hawker:    [np.float64(0.1724193099680737), np.float64(0.4105462439026658), np.float64(0.5855211609152101)]
dist_to_cbd:            [8.789584168467353, 9.889190833168252, 11.008129126934422]
num_mrt_within_1km:     [np.int64(0), np.int64(0), np.int64(0)]
num_hawkers_within_500m:[np.int64(1), np.int64(3), np.int64(0)]


In [25]:
df['dist_nearest_mrt'] = dist_nearest_mrt
df['num_mrt_within_1km'] = num_mrt_within_1km
df['num_mrt_within_2km'] = num_mrt_within_2km
df['dist_nearest_school'] = dist_nearest_school
df['num_schools_within_1km'] = num_schools_within_1km
df['dist_nearest_primary_school'] = dist_nearest_primary_school
df['num_primary_schools_within_1km'] = num_primary_schools_within_1km
df['dist_nearest_mall'] = dist_nearest_mall
df['num_malls_within_2km'] = num_malls_within_2km
df['dist_nearest_hawker'] = dist_nearest_hawker
df['num_hawkers_within_500m'] = num_hawkers_within_500m
df['dist_to_cbd'] = dist_to_cbd
df['dist_nearest_expressway'] = dist_nearest_expressway
df['dist_nearest_bus_stop'] = dist_nearest_bus_stop
df['num_bus_stops_within_300m'] = num_bus_stops_within_300m

print("Features added.")
print(f"DataFrame shape: {df.shape}")
print(f"\nNew columns:")
print(df[['dist_nearest_mrt', 'dist_to_cbd', 'dist_nearest_hawker']].describe())

Features added.
DataFrame shape: (233055, 29)

New columns:
       dist_nearest_mrt    dist_to_cbd  dist_nearest_hawker
count     233055.000000  233055.000000         2.330550e+05
mean           0.792527      12.515402         7.499634e-01
std            0.443371       4.410231         5.142920e-01
min            0.036138       0.591812         2.526184e-08
25%            0.466696       9.778165         3.543361e-01
50%            0.707175      13.435929         6.350460e-01
75%            1.028425      15.515399         1.008811e+00
max            3.516004      20.312713         2.864821e+00


In [26]:
mature_estates = [
    'ANG MO KIO', 'BEDOK', 'BISHAN', 'BUKIT MERAH', 'BUKIT TIMAH',
    'CENTRAL AREA', 'CLEMENTI', 'GEYLANG', 'KALLANG/WHAMPOA',
    'MARINE PARADE', 'PASIR RIS', 'QUEENSTOWN', 'SERANGOON',
    'TAMPINES', 'TOA PAYOH'
]

df['is_mature_estate'] = df['town'].isin(mature_estates).astype(int)

print(f"Mature estate transactions: {df['is_mature_estate'].sum():,}")
print(f"Non-mature estate transactions: {(df['is_mature_estate'] == 0).sum():,}")

Mature estate transactions: 97,253
Non-mature estate transactions: 135,802


In [27]:
df.to_csv('../data/processed/features_with_geo.csv', index=False)
print(f"Saved to data/processed/features_with_geo.csv")
print(f"Shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")

Saved to data/processed/features_with_geo.csv
Shape: (233055, 30)

Columns: ['month', 'town', 'flat_type', 'block', 'street_name', 'storey_range', 'floor_area_sqm', 'flat_model', 'lease_commence_date', 'remaining_lease', 'resale_price', 'latitude', 'longitude', 'transaction_date', 'dist_nearest_mrt', 'num_mrt_within_1km', 'num_mrt_within_2km', 'dist_nearest_school', 'num_schools_within_1km', 'dist_nearest_primary_school', 'num_primary_schools_within_1km', 'dist_nearest_mall', 'num_malls_within_2km', 'dist_nearest_hawker', 'num_hawkers_within_500m', 'dist_to_cbd', 'dist_nearest_expressway', 'dist_nearest_bus_stop', 'num_bus_stops_within_300m', 'is_mature_estate']
